# Category-aware reranking po korekcji hubness

Ten notebook jest następnym krokiem po `KOREKCJA_HUBNESS_RERANKING_COLAB.ipynb`.

Poprzedni wynik pokazał dwie rzeczy:

- `candidate_zscore` ogranicza hubness i poprawia retrieval (`SSIM ≈ 0.264`),
- oracle prawdziwej kategorii ma bardzo wysoki potencjał (`SSIM ≈ 0.431`).

Hipoteza tego notebooka: jeśli najpierw użyjemy EEGNet jako bramki kategorii, a dopiero potem zrobimy reranking obrazów w przestrzeni EEG→CLIP, to rekonstrukcja typu candidate-constrained może przebić VAE baseline.

## Co dokładnie jest robione

Notebook używa trzech źródeł informacji:

1. `unclip_mole_retrieval/eeg_image_retrieval.pt` — model EEG→CLIP embedding.
2. `eegnet_mole_colab/eegnet.pt` — model EEGNet przewidujący kategorię obrazu z EEG.
3. `image_embeddings_unclip_participant_image_mole_no_abc_local_20260627` — embeddingi obrazów kompatybilne z UnCLIP/CLIP.

Dla każdego obrazu testowego uśredniamy powtórzenia EEG, liczymy score do kandydatów i testujemy warianty:

- `candidate_zscore` jako baseline po korekcji hubness,
- self-category priors wyciągnięte z samych score'ów CLIP,
- `eegnet_topK_category_gate`, czyli ograniczenie kandydatów do kategorii przewidzianych przez EEGNet,
- `eegnet_zlogprob_alpha_*`, czyli miękkie dodanie prioru kategorii EEGNet,
- oracle prawdziwej kategorii tylko jako górną granicę, nie jako uczciwy wynik.

Metoda finalna jest wybierana na walidacji, a potem raportowana na teście.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'scripts').is_dir() else CWD.parent
OUTPUT_DIR = PROJECT_ROOT / 'wyniki colab' / 'category_aware_reranking_mole_local'

SCRIPT = PROJECT_ROOT / 'scripts' / 'run_category_aware_reranking.py'
MANIFEST_DIR = PROJECT_ROOT / 'reconstruction_manifests' / 'participant_image_mole_no_abc'
EMBEDDING_DIR = PROJECT_ROOT / 'image_embeddings_unclip_participant_image_mole_no_abc_local_20260627'
RETRIEVAL_CHECKPOINT = PROJECT_ROOT / 'wyniki colab' / 'unclip_mole_retrieval' / 'eeg_image_retrieval.pt'
EEGNET_CHECKPOINT = PROJECT_ROOT / 'wyniki colab' / 'eegnet_mole_colab' / 'eegnet.pt'

for path in [SCRIPT, MANIFEST_DIR, EMBEDDING_DIR, RETRIEVAL_CHECKPOINT, EEGNET_CHECKPOINT]:
    print(path, 'OK' if path.exists() else 'BRAK')

In [ ]:
# Uruchom pełny lokalny eksperyment.
# --force czyści tylko OUTPUT_DIR tego eksperymentu, żeby nie mieszać starych częściowych wyników.
import subprocess, sys

cmd = [
    sys.executable, str(SCRIPT),
    '--project-root', str(PROJECT_ROOT),
    '--manifest-dir', str(MANIFEST_DIR),
    '--embedding-dir', str(EMBEDDING_DIR),
    '--retrieval-checkpoint', str(RETRIEVAL_CHECKPOINT),
    '--eegnet-checkpoint', str(EEGNET_CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Podsumowanie wybranej metody i porównanie z post-hoc najlepszą metodą na teście.
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / 'category_aware_reranking_summary.json').read_text(encoding='utf-8'))
selected = pd.read_csv(OUTPUT_DIR / 'selected_method_comparison.csv')
display(selected)

print('Wybrana metoda:', summary['selected_method'])
print('Test SSIM:', summary['test_selected_by_validation']['ssim'])
print('Test top1:', summary['test_selected_by_validation']['top1'])
print('Test top5:', summary['test_selected_by_validation']['top5'])
print('Test category_top1:', summary['test_selected_by_validation']['category_top1'])

In [ ]:
# Ranking metod na walidacji i teście.
for split in ['validation', 'test']:
    df = pd.read_csv(OUTPUT_DIR / split / 'method_comparison.csv')
    cols = ['method', 'top1', 'top5', 'top10', 'category_top1', 'ssim', 'distinct_predictions']
    print('\n' + split.upper())
    display(df.sort_values(['is_oracle', 'ssim', 'top5'], ascending=[True, False, False])[cols].head(12))

    diag = pd.read_csv(OUTPUT_DIR / split / 'category_diagnostics.csv')
    print('Diagnostyka kategorii:')
    display(diag)

In [ ]:
# Podgląd gridów testowych.
from IPython.display import Image as IPImage, Markdown, display

for path in sorted((OUTPUT_DIR / 'grids').glob('*.jpg')):
    display(Markdown(f'### {path.name}'))
    display(IPImage(filename=str(path)))

## Wpis historyczny — wynik lokalny z 2026-06-27

Lokalny run wskazał `eegnet_top1_category_gate` jako metodę wybraną na walidacji. Na teście metoda osiągnęła:

- `top1 = 18.18%`,
- `top5 = 38.64%`,
- `category_top1 = 38.64%`,
- `SSIM = 0.308`.

To przebija dotychczasowy VAE ensemble dla `mole` (`SSIM ≈ 0.286`) oraz poprzedni najlepszy wariant bez kategorii `candidate_zscore` (`SSIM ≈ 0.264`).

Trzeba jednak uczciwie nazwać zakres: to nie jest jeszcze wolna generacja obrazu z EEG. To jest **candidate-constrained reconstruction** — model wybiera najlepszy obraz z puli kandydatów. Mimo tego jest to ważny krok, bo pokazuje, że połączenie sygnału kategorii z EEGNet i embeddingowego rerankingu daje silniejszy wynik niż każdy z tych elementów osobno.